# 🏆 Capstone Project: The Used-Car Price Estimator
**Week 7 · Notebook 10 of 10 · ⏱️ 4–6 hours (spread over several sittings)**

---

> *"In theory there is no difference between theory and practice. In practice there is."*

You have learned the ideas. Now you'll face **real-world messiness**: missing values, text categories, several candidate models, a modelling insight you must discover for yourself, and a report that a decision-maker could act on. This is a *project*, not a worksheet: expect to think, to get stuck, and to iterate.

## 📋 The scenario

**RoadRunner Motors** is a used-car dealership. Today, an experienced appraiser prices each trade-in by gut feeling, and prices vary wildly between appraisers. The owner wants a **data-driven price estimator** that:

1. gives a *fair first quote* within seconds,
2. is **honest about its typical error**, and
3. tells the team **where it can't be trusted**, so a human appraiser can step in.

**You** are the data scientist. Build it, evaluate it rigorously, and write the report.

## 🎯 What you will demonstrate

| Skill | Where |
|---|---|
| Exploring a messy dataset | Milestone 1 |
| Honest splitting & baselines | Milestone 2 |
| Pipelines: imputation, encoding, scaling: **without leakage** | Milestone 3 |
| Model comparison with **cross-validation** | Milestone 4 |
| Diagnosing a model & **improving it by thinking** (target transformation) | Milestone 5 |
| Hyperparameter tuning & regularisation | Milestone 6 |
| Final, single-shot test evaluation with the right metrics | Milestone 7 |
| Error analysis: *where* does it fail and *why*? | Milestone 8 |
| Communicating results to non-experts | Milestone 9 |

## 🧰 The dataset: `used_cars.csv` (1,200 listings)

| Column | Meaning | Notes |
|---|---|---|
| `brand` | Toyota, Honda, Ford, Kia, BMW | text category |
| `year` | year of manufacture | |
| `mileage_km` | kilometres driven | **has missing values** |
| `engine_cc` | engine size in cc | **has missing values** |
| `fuel_type` | petrol, diesel, hybrid | text category |
| `transmission` | manual, automatic | text category |
| `owners` | number of previous owners | **has missing values** |
| **`price_usd`** | **asking price in US dollars** | **TARGET** |

## 📏 Grading rubric (100 points)

| Criterion | Pts | Excellent looks like |
|---|---|---|
| **Data understanding** (M1) | 10 | missing values found, relationships plotted, at least one real insight |
| **Methodology** (M2–M4, M6) | 25 | correct split, baseline, leakage-free pipeline, model comparison via CV, tuning on training data only |
| **Modelling insight** (M5) | 10 | recognises *why* a transformation helps and shows it with evidence |
| **Final evaluation** (M7) | 15 | test set used **once**; MAE/RMSE/R²/MAPE; baseline comparison; residual plot |
| **Error analysis** (M8) | 15 | identifies where the model is weak and offers a plausible reason |
| **Report** (M9) | 20 | clear, quantitative, honest, actionable, jargon-free |
| **Code quality & reproducibility** | 5 | runs top-to-bottom; `random_state` set; readable |

## ⚖️ Ground rules

1. **The test set is touched exactly once** (Milestone 7). All model selection uses cross-validation on the training set.
2. **No leakage.** All preprocessing is learned inside a `Pipeline`.
3. **Every claim has a number.** "The model is good" is not a result; "typical error is $634 (7%), 80% better than the baseline" is.
4. Hints are there to be used, but one at a time. 😉

In [1]:
import sys
sys.path.append("..")
from course_utils import *
setup()

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import clone
from sklearn.model_selection import train_test_split, cross_val_score, KFold, GridSearchCV
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.dummy import DummyRegressor
from sklearn.metrics import (mean_absolute_error, mean_squared_error, r2_score, mean_absolute_percentage_error)

def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

---
# 📖 Primer: Preprocessing messy data the leakage-proof way

Your earlier datasets were clean numeric tables. Real data has two problems that models can't digest:

1. **Missing values (`NaN`)**: most scikit-learn models crash on them.
2. **Text categories** (`"BMW"`, `"diesel"`): models need numbers.

The solutions are standard tools. (They come from the next phase of your studies; here is everything you need.)

| Problem | Tool | What it does |
|---|---|---|
| Missing numbers | `SimpleImputer(strategy="median")` | fills gaps with the column's median (robust to outliers) |
| Missing categories | `SimpleImputer(strategy="most_frequent")` | fills with the most common value |
| Text categories | `OneHotEncoder(handle_unknown="ignore")` | one 0/1 column per category (`brand_BMW`, `brand_Kia`, …) |
| Mixed scales | `StandardScaler()` | mean 0, std 1 (Lessons 3, 5, 8) |
| Apply *different* steps to *different* columns | `ColumnTransformer` | e.g. numbers → impute + scale; text → impute + one-hot |
| Chain preprocessing and a model into **one** object | `Pipeline` | `fit` learns *everything* from training data only → **no leakage** |

> 🧠 **Why one-hot instead of 1, 2, 3, 4, 5?** If we coded brands as Toyota=1, Honda=2, Ford=3, … a linear model would think Ford is "three times" Toyota. One-hot columns let each brand have its *own* effect.

### 🧑‍🏫 Worked Example 1: Imputing and encoding a toy table

In [2]:
toy = pd.DataFrame({"age_years": [3, 10, np.nan, 5], "fuel": ["petrol", "diesel", "petrol", np.nan]})
print("Original:"); display(toy)

num_steps = make_pipeline(SimpleImputer(strategy="median"), StandardScaler())
cat_steps = make_pipeline(SimpleImputer(strategy="most_frequent"), OneHotEncoder(handle_unknown="ignore"))

print("Numeric column after impute (NaN → median 5) + scale:")
print(num_steps.fit_transform(toy[["age_years"]]).round(2).ravel())
print("\nCategory column after impute (NaN → 'petrol') + one-hot:")
print(pd.DataFrame(cat_steps.fit_transform(toy[["fuel"]]).toarray(),
                   columns=cat_steps[-1].get_feature_names_out()))

Original:


,age_years,fuel
0,3.0,petrol
1,10.0,diesel
2,NaN,petrol
3,5.0,NaN


Numeric column after impute (NaN → median 5) + scale:
[-1.06  1.64 -0.29 -0.29]

Category column after impute (NaN → 'petrol') + one-hot:
   x0_diesel  x0_petrol
0        0.0        1.0
1        1.0        0.0
2        0.0        1.0
3        0.0        1.0


The median of [3, 10, 5] is **5**, so the missing age becomes 5 before scaling. The most frequent fuel is `petrol`, so the missing fuel becomes `petrol` and then splits into `fuel_diesel` / `fuel_petrol` columns.

### 🧑‍🏫 Worked Example 2: One object that does it all

A `ColumnTransformer` routes columns to the right treatment, and a `Pipeline` bolts a model on the end. Calling `.fit(X_train, y_train)` learns the medians, categories, scalings **and** the regression weights: all from the training rows only.

In [3]:
toy_X = pd.DataFrame({"age_years": [3, 10, np.nan, 5, 8, 1], "fuel": ["petrol", "diesel", "petrol", np.nan, "diesel", "petrol"]})
toy_y = [20, 8, 15, 14, 9, 25]

preprocess_demo = ColumnTransformer([
    ("numbers", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), ["age_years"]),
    ("text",    make_pipeline(SimpleImputer(strategy="most_frequent"), OneHotEncoder(handle_unknown="ignore")), ["fuel"]),
])
demo_model = make_pipeline(preprocess_demo, LinearRegression()).fit(toy_X, toy_y)

# It even copes with NEW, messy data at prediction time:
new_car = pd.DataFrame({"age_years": [np.nan], "fuel": ["hybrid"]})      # missing age AND an unseen fuel type!
print("prediction for a messy new car:", demo_model.predict(new_car).round(2))

prediction for a messy new car: [16.38]


That robustness (missing values *and* an unseen category at prediction time) is exactly what a real dealership system needs.

> 🎒 **You now have the toolkit. Let's begin the project.**

### 🧩 Milestone 10.1: Understand the data &nbsp; ⭐⭐

Before modelling, **explore**. The data is loaded for you in `cars`.

1. `missing_counts`: a pandas Series with the **number of missing values for each column that has any** (skip columns with none).
2. `priciest_brand`: the brand with the highest **median** `price_usd`.
3. `corr_with_price`: a Series of the correlation of `year`, `mileage_km`, `engine_cc`, `owners` with `price_usd`.
4. **Make two plots** (not auto-checked, but your report will need them): (a) price vs. year, (b) median price by brand (bar chart). What patterns do you see? Is the price distribution symmetric or skewed?

**Stuck?** Open the hints one at a time: try again between each one.

<details>
<summary>💡 Hint 1</summary>

`cars.isna().sum()` counts missing values per column; filter with `s[s > 0]`.

</details>

<details>
<summary>💡 Hint 2</summary>

`cars.groupby("brand")["price_usd"].median().idxmax()`.

</details>

<details>
<summary>💡 Hint 3</summary>

`cars[["year", "mileage_km", "engine_cc", "owners", "price_usd"]].corr()["price_usd"]`: then drop the `price_usd` row (correlation with itself).

</details>

In [ ]:
cars = load("used_cars.csv")          # (given)

missing_counts = None
priciest_brand = None
corr_with_price = None

# plots here

In [ ]:
# ✔ Self-check - run this after you have solved the problem
check("three columns have missing values", lambda: set(missing_counts.index) == {"mileage_km", "engine_cc", "owners"})
check("missing counts are 41 / 30 / 21", lambda: missing_counts["mileage_km"] == 41 and missing_counts["engine_cc"] == 30 and missing_counts["owners"] == 21)
check("BMW is the priciest brand", lambda: priciest_brand == "BMW")
check("newer cars cost more (corr(year) > 0.5)", lambda: corr_with_price["year"] > 0.5)
check("higher mileage means lower price (corr < 0)", lambda: corr_with_price["mileage_km"] < 0)

### 🧩 Milestone 10.2: Split the data and set a baseline &nbsp; ⭐⭐

1. Separate `X` (all columns except `price_usd`) and `y` (`price_usd`), and split **80/20** with `random_state=0` → `X_train, X_test, y_train, y_test`.
2. Fit a `DummyRegressor(strategy="mean")` on the **training** data and compute its **test** RMSE (`baseline_rmse`) and MAE (`baseline_mae`).

*This baseline is the number every model must beat. Write it down: you will quote it in your report.*

**Stuck?** Open the hints one at a time: try again between each one.

<details>
<summary>💡 Hint 1</summary>

`train_test_split(X, y, test_size=0.2, random_state=0)`.

</details>

<details>
<summary>💡 Hint 2</summary>

`DummyRegressor` ignores features: it can be fitted on `X_train` even with missing values.

</details>

In [ ]:
X = cars.drop(columns="price_usd")      # (given)
y = cars["price_usd"]                   # (given)

X_train = X_test = y_train = y_test = None
baseline_rmse = baseline_mae = None

In [ ]:
# ✔ Self-check - run this after you have solved the problem
check("960 train / 240 test rows", lambda: len(X_train) == 960 and len(X_test) == 240)
check("split is reproducible (same as random_state=0)", lambda: list(X_test.index) == list(train_test_split(X, y, test_size=0.2, random_state=0)[1].index))
check("baseline RMSE ≈ 6,687", lambda: abs(baseline_rmse - 6686.6) < 1)
check("baseline MAE ≈ 4,862", lambda: abs(baseline_mae - 4861.9) < 1)

### 🧩 Milestone 10.3: Build a leakage-proof preprocessing pipeline &nbsp; ⭐⭐⭐

Using the toolkit from the primer:

1. Define `num_cols = ["year", "mileage_km", "engine_cc", "owners"]` and `cat_cols = ["brand", "fuel_type", "transmission"]`.
2. Build `preprocess`, a `ColumnTransformer` that
   * sends numeric columns through `SimpleImputer(strategy="median")` then `StandardScaler()`,
   * sends categorical columns through `SimpleImputer(strategy="most_frequent")` then `OneHotEncoder(handle_unknown="ignore")`.
3. Build `lin_pipe = make_pipeline(preprocess, LinearRegression())`, **fit it on the training data**, and store its **test RMSE** in `lin_test_rmse`.

**Stuck?** Open the hints one at a time: try again between each one.

<details>
<summary>💡 Hint 1</summary>

Follow Worked Example 2: `ColumnTransformer([("numbers", <pipeline>, num_cols), ("text", <pipeline>, cat_cols)])`.

</details>

<details>
<summary>💡 Hint 2</summary>

Each branch is itself a `make_pipeline(imputer, scaler_or_encoder)`.

</details>

<details>
<summary>💡 Hint 3</summary>

`lin_pipe.fit(X_train, y_train)`, then `rmse(y_test, lin_pipe.predict(X_test))`. (Peeking at the test set once here is acceptable as a *sanity check*; model selection in M4 must use cross-validation.)

</details>

In [ ]:
num_cols = ["year", "mileage_km", "engine_cc", "owners"]
cat_cols = ["brand", "fuel_type", "transmission"]

preprocess = None
lin_pipe = None
lin_test_rmse = None

In [ ]:
# ✔ Self-check - run this after you have solved the problem
check("preprocess is a ColumnTransformer", lambda: isinstance(preprocess, ColumnTransformer))
check("the pipeline is fitted and predicts without errors on data with NaNs", lambda: len(lin_pipe.predict(X_test)) == 240)
check("one-hot columns appear: 4 numeric + 5 brands + 3 fuels + 2 transmissions = 14", lambda: lin_pipe[0].transform(X_test).shape[1] == 14)
check("beats the baseline by a wide margin (RMSE < 50% of baseline)", lambda: lin_test_rmse < 0.5 * baseline_rmse)

### 🧩 Milestone 10.4: Compare models with cross-validation &nbsp; ⭐⭐⭐

Compare four model families using **5-fold cross-validation on the training set only**, with `KFold(n_splits=5, shuffle=True, random_state=0)` and `scoring="neg_root_mean_squared_error"`.

Each model must be wrapped in its own pipeline with a **clone of `preprocess`**:

| Name | Model |
|---|---|
| `"Linear"` | `LinearRegression()` |
| `"Ridge"` | `Ridge(alpha=1)` |
| `"KNN"` | `KNeighborsRegressor(n_neighbors=10)` |
| `"Tree"` | `DecisionTreeRegressor(max_depth=6, random_state=0)` |

Store the **mean CV RMSE** (positive numbers!) in `cv_rmse` (`{name: rmse}`) and the best model's name in `best_name`.

**Stuck?** Open the hints one at a time: try again between each one.

<details>
<summary>💡 Hint 1</summary>

`make_pipeline(clone(preprocess), model)`.

</details>

<details>
<summary>💡 Hint 2</summary>

`-cross_val_score(pipe, X_train, y_train, cv=kf, scoring="neg_root_mean_squared_error").mean()` gives a positive RMSE.

</details>

<details>
<summary>💡 Hint 3</summary>

`best_name = min(cv_rmse, key=cv_rmse.get)`: lowest error wins.

</details>

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=0)

cv_rmse = {}
best_name = None
print(cv_rmse, best_name)

In [ ]:
# ✔ Self-check - run this after you have solved the problem
check("four models evaluated", lambda: sorted(cv_rmse) == ["KNN", "Linear", "Ridge", "Tree"])
check("best_name is the model with the lowest CV RMSE", lambda: best_name == min(cv_rmse, key=cv_rmse.get))
check("the non-linear Tree beats Linear (the data is not linear in price)", lambda: cv_rmse["Tree"] < cv_rmse["Linear"])
check("Linear and Ridge are practically identical (within 2%)", lambda: abs(cv_rmse["Linear"] - cv_rmse["Ridge"]) / cv_rmse["Linear"] < 0.02)

### 🧩 Milestone 10.5: Improve the model by thinking: transform the target &nbsp; ⭐⭐⭐

*Clue from Milestone 1:* depreciation is **proportional**: a car loses a *percentage* of its value each year, and prices are right-skewed. When effects **multiply**, taking **logarithms** turns them into effects that **add**: which is exactly what a linear model handles.

Wrap the linear pipeline so that the model learns **log(price)** but predicts in **dollars**:

```python
TransformedTargetRegressor(regressor=<your linear pipeline>, func=np.log, inverse_func=np.exp)
```

Compute its 5-fold CV RMSE (same `kf`, **in dollars**, positive) → `cv_rmse_log`. Store the relative improvement over the plain linear model in `gain_pct = 100 * (cv_rmse["Linear"] − cv_rmse_log) / cv_rmse["Linear"]`.

**Stuck?** Open the hints one at a time: try again between each one.

<details>
<summary>💡 Hint 1</summary>

`TransformedTargetRegressor(regressor=make_pipeline(clone(preprocess), LinearRegression()), func=np.log, inverse_func=np.exp)`.

</details>

<details>
<summary>💡 Hint 2</summary>

It behaves like any estimator: pass it to `cross_val_score` with the same scoring as before. Errors are still measured in **dollars**, because predictions are transformed back.

</details>

In [ ]:
log_linear = None
cv_rmse_log = None
gain_pct = None
print(cv_rmse_log, gain_pct)

In [ ]:
# ✔ Self-check - run this after you have solved the problem
check("a TransformedTargetRegressor was built", lambda: isinstance(log_linear, TransformedTargetRegressor))
check("log-target linear model has CV RMSE < 1,300", lambda: cv_rmse_log < 1300)
check("it beats the non-linear Tree too", lambda: cv_rmse_log < cv_rmse["Tree"])
check("improvement over plain linear is > 50%", lambda: gain_pct > 50)

### 🧩 Milestone 10.6: Tune hyperparameters (on training data only!) &nbsp; ⭐⭐⭐

Tune the **Ridge** strength for the log-target model with `GridSearchCV`:

* estimator: a `TransformedTargetRegressor` wrapping `make_pipeline(clone(preprocess), Ridge())`, `func=np.log`, `inverse_func=np.exp`;
* parameter grid: the Ridge `alpha` in `[0.01, 0.1, 1, 10, 100]`: the parameter name is **`"regressor__ridge__alpha"`** (outer step `regressor`, then the pipeline's step `ridge`);
* `cv=kf`, `scoring="neg_root_mean_squared_error"`, fitted on the **training set**.

Store the fitted search in `grid`, the winning alpha in `best_alpha`, and the winning CV RMSE (positive) in `best_cv_rmse`.

**Stuck?** Open the hints one at a time: try again between each one.

<details>
<summary>💡 Hint 1</summary>

`make_pipeline` names each step after its class in lowercase: `ridge`.

</details>

<details>
<summary>💡 Hint 2</summary>

`grid = GridSearchCV(estimator, {"regressor__ridge__alpha": [...]}, cv=kf, scoring="neg_root_mean_squared_error").fit(X_train, y_train)`.

</details>

<details>
<summary>💡 Hint 3</summary>

`grid.best_params_["regressor__ridge__alpha"]` and `-grid.best_score_`.

</details>

In [ ]:
grid = None
best_alpha = None
best_cv_rmse = None
print(best_alpha, best_cv_rmse)

In [ ]:
# ✔ Self-check - run this after you have solved the problem
check("grid search was run on 5 alphas", lambda: len(grid.cv_results_["params"]) == 5)
check("best_alpha is one of the candidates", lambda: best_alpha in [0.01, 0.1, 1, 10, 100])
check("tuned CV RMSE < 1,150", lambda: best_cv_rmse < 1150)
check("very strong regularisation (α = 100) was NOT best", lambda: best_alpha != 100)

### 🧩 Milestone 10.7: The moment of truth: final test evaluation &nbsp; ⭐⭐⭐

**Open the sealed envelope: once.** Take the tuned model `grid.best_estimator_` and evaluate it on the **test set**:

* `final_metrics`: a dictionary with keys `"MAE"`, `"RMSE"`, `"R2"`, `"MAPE"` (as a fraction, e.g. 0.07 = 7%);
* `improvement_pct`: % reduction of the test RMSE relative to `baseline_rmse`.

Also draw a **predicted-vs-actual** plot and a **residual plot** for the test set.

**Stuck?** Open the hints one at a time: try again between each one.

<details>
<summary>💡 Hint 1</summary>

`final_model = grid.best_estimator_`; `pred = final_model.predict(X_test)`.

</details>

<details>
<summary>💡 Hint 2</summary>

`mean_absolute_percentage_error(y_test, pred)` returns a fraction.

</details>

<details>
<summary>💡 Hint 3</summary>

`improvement_pct = 100 * (baseline_rmse - final_metrics["RMSE"]) / baseline_rmse`.

</details>

In [ ]:
final_model = None
final_pred = None
final_metrics = {}
improvement_pct = None

# plots below

In [ ]:
# ✔ Self-check - run this after you have solved the problem
check("metrics dictionary has the four keys", lambda: sorted(final_metrics) == ["MAE", "MAPE", "R2", "RMSE"])
check("R² is above 0.9", lambda: final_metrics["R2"] > 0.9)
check("test RMSE is below 25% of the baseline's", lambda: final_metrics["RMSE"] < 0.25 * baseline_rmse)
check("typical percentage error (MAPE) is under 10%", lambda: final_metrics["MAPE"] < 0.10)
check("improvement_pct is consistent with the metrics", lambda: abs(improvement_pct - 100 * (baseline_rmse - final_metrics["RMSE"]) / baseline_rmse) < 1e-6)

### 🧩 Milestone 10.8: Error analysis: where does it fail? &nbsp; ⭐⭐⭐

A model you can't *criticise* is a model you don't understand. Using the final test predictions:

1. `worst`: a DataFrame of the **10 test cars with the largest absolute error**, with columns `brand, year, mileage_km, engine_cc, actual, predicted, abs_error` (in dollars), sorted from worst to less bad.
2. `mape_by_brand`: a Series: the **mean absolute percentage error** of the test predictions for each `brand` (index = brand).
3. `hardest_brand`: the brand with the highest `mape_by_brand`.

*Then investigate:* what do the worst cars have in common (newer? pricier? a particular brand? **missing values** that were imputed?)

**Stuck?** Open the hints one at a time: try again between each one.

<details>
<summary>💡 Hint 1</summary>

Build a DataFrame from `X_test` and add `actual`, `predicted` and `abs_error` columns, then `.sort_values("abs_error", ascending=False).head(10)`.

</details>

<details>
<summary>💡 Hint 2</summary>

`ape = np.abs(y_test - final_pred) / y_test`; then `ape.groupby(X_test["brand"]).mean()`.

</details>

In [ ]:
worst = None
mape_by_brand = None
hardest_brand = None
print(worst); print(mape_by_brand)

In [ ]:
# ✔ Self-check - run this after you have solved the problem
check("10 rows", lambda: len(worst) == 10)
check("required columns", lambda: list(worst.columns) == ["brand", "year", "mileage_km", "engine_cc", "actual", "predicted", "abs_error"])
check("sorted worst → less bad, starting at the largest error", lambda: worst["abs_error"].is_monotonic_decreasing and abs(worst["abs_error"].iloc[0] - np.abs(y_test.values - final_pred).max()) < 1e-6)
check("MAPE by brand for all five brands", lambda: sorted(mape_by_brand.index) == ["BMW", "Ford", "Honda", "Kia", "Toyota"])
check("hardest_brand matches the table", lambda: hardest_brand == mape_by_brand.idxmax())

### 🧩 Milestone 10.9: Write the report &nbsp; ⭐⭐⭐

Write a one-page report for the dealership owner. Complete the dictionary `report` (**at least 80 characters per field**; quote numbers):

* `problem`: what you built and why it matters;
* `data`: what data, its size, its problems (missing values!) and how you handled them;
* `method`: what you tried, in order, and why you chose the final model (include the CV evidence and the log-target insight);
* `results`: test-set MAE / RMSE / R² / MAPE **and** the comparison with the baseline;
* `error_analysis`: where it fails and what to do about it;
* `limitations`: at least two honest caveats (data, evaluation, time);
* `recommendation`: what the dealership should do next.

**Reference your own numbers**: scroll up to read them off your outputs.

**Stuck?** Open the hints one at a time: try again between each one.

<details>
<summary>💡 Hint 1</summary>

Structure each field as *claim → evidence (a number) → implication*. E.g. "The model is typically off by $630 (7%)… so quotes can be automated for ordinary cars."

</details>

<details>
<summary>💡 Hint 2</summary>

For `limitations`: one data limitation (only 1,200 listings; asking prices ≠ sale prices; no condition or region) and one evaluation limitation (a single 240-car test set; no check on future market changes).

</details>

In [ ]:
report = {
    "problem": "",
    "data": "",
    "method": "",
    "results": "",
    "error_analysis": "",
    "limitations": "",
    "recommendation": "",
}

In [ ]:
# ✔ Self-check - run this after you have solved the problem
_fields = ["problem", "data", "method", "results", "error_analysis", "limitations", "recommendation"]
check("seven fields present", lambda: sorted(report) == sorted(_fields))
for _f in _fields:
    check(f"'{_f}' has at least 80 characters", lambda: len(report[_f].strip()) >= 80)
check("the report cites plenty of numbers (≥ 25 digits)", lambda: sum(ch.isdigit() for ch in " ".join(report.values())) >= 25)

---
# 🚀 Stretch goals (choose any: all optional)

| # | Challenge | What you'll practise |
|---|---|---|
| 1 | **Feature engineering.** Add `age = 2024 − year` and `km_per_year = mileage_km / age` (handle age 0!). Does CV RMSE improve? | creating informative features |
| 2 | **Learning curves.** Plot CV error vs training-set size for the final model. Would more data help? | diagnosing bias vs variance (Lesson 5) |
| 3 | **Classification twist.** Create `expensive = price_usd > median` and train a `LogisticRegression` pipeline. Report precision, recall, F1, ROC-AUC. Which threshold would you choose for an *"auto-approve cheap trade-ins"* policy? | Lesson 7 end-to-end |
| 4 | **Segment the market.** Run K-Means on scaled numeric features; profile the clusters. Do clusters match brands? | unsupervised learning (Lesson 3) |
| 5 | **Quantify uncertainty.** Fit the model to 200 bootstrap resamples of the training set; give a ±range for a quoted price. | variance in action (Lesson 5) |
| 6 | **Own the optimiser.** Implement gradient descent for the log-price linear model (on the preprocessed matrix) and compare coefficients with scikit-learn. | Lesson 8 |
| 7 | **Flag the unknown.** Use `SimpleImputer(..., add_indicator=True)` so the model knows which values were missing. Do the worst errors from Milestone 8 shrink? | handling missing data honestly |
| 8 | **Beat the tree on the log scale.** Fit a `DecisionTreeRegressor` inside a `TransformedTargetRegressor`; tune `max_depth` with CV. Does the tree catch up? | model comparison |

# ✅ Final submission checklist

- [ ] *Restart & Run All* works with no errors.
- [ ] The test set is used **exactly once** (Milestone 7).
- [ ] All preprocessing sits inside a pipeline fitted on training data only.
- [ ] I compared at least four models with cross-validation and justified my choice with numbers.
- [ ] My report includes: baseline, final metrics, where the model fails, honest limitations, a recommendation.
- [ ] A non-technical reader could understand my report.

---

# 🎓 Graduation

Take a moment to appreciate how far you've come. Three weeks ago, "machine learning" was a phrase. Now you can:

* explain **what** a model is and **how** it learns (loss, gradient descent, parameters, hyperparameters);
* **frame** a problem as regression, classification or clustering;
* **split** data honestly, **avoid leakage**, and **evaluate** with the right metric;
* **diagnose** underfitting and overfitting, and **fix** them with regularisation and cross-validation;
* **build, compare, tune and criticise** a complete model, and **explain** it to a non-expert.

That is the foundation on which *everything* else (trees and forests, neural networks, deep learning, language models) is built. They are all variations on the loop you now understand:

> **data → model → loss → gradient → update → evaluate honestly → explain.**

Go build something wonderful. 🚀